# Text to command

In [ ]:
# !pip install openai

from openai import OpenAI
from enum import Enum
import pandas as pd
import pickle
import time
import json
import os
import sys
sys.path.insert(1, '../')
from common import get_test_dataset, is_command_classification_correct, get_prompt_template

# set environment variable OPENAI_API_KEY, see: https://platform.openai.com/api-keys
os.environ["OPENAI_API_KEY"] =

# Get the test dataset 
# X - prompts, y - raw commands
X , y = get_test_dataset('../data/basic_commands_v1.0.json')

### Example prompt with speed request

In [2]:
print(X[75][0])

Increasing speed to 50 m/s


In [3]:
print(X[75][1])

There is a set of predefined commands:

STOP – Halts the vehicle immediately.
START – Starts driving.
ENGINE_STOP - Turns off the vehicle’s engine.
ENGINE_START - Turns on the vehicle’s engine.
CRANK_REQUEST - Initiates the engine cranking process.
STOP_IN_PITLANE_TOGGLE_ON - Directs the vehicle to enter the pit lane and stop there.
STOP_IN_PITLANE_TOGGLE_OFF - Cancels the pit lane stop mode, allowing the vehicle to resume normal driving.
SPEED_BY_RC_TOGGLE_ON - Enables race control to manage the vehicle’s speed, preventing manual control by the driver.
SPEED_BY_RC_TOGGLE_OFF - Disables race control's speed management, allowing the driver to control the vehicle’s speed manually.
FLAG_BY_RC_TOGGLE_ON - Activates race control flag signaling to the vehicle, preventing the driver from manually controlling flags.
FLAG_BY_RC_TOGGLE_OFF - Disables race control flag signaling, allowing the driver to manage flags manually.
INTO_PITLANE_TOGGLE_ON - Directs the vehicle to enter the pit lane, but 

In [4]:
print(f"Request: {y[75][0]}, value: {y[75][1]} m/s")

Request: SPEED_REQUEST, value: 50 m/s


### OpenAI models

In [5]:
client = OpenAI()

completion = client.chat.completions.create(
    model="chatgpt-4o-latest",
    response_format={"type": "json_object"},
    messages=[
    {"role": "system",  "content": get_prompt_template()},
    {"role": "user",  "content": X[75][0]}
    ]
)

answer = completion.choices[0].message.content
print(answer)

{
    "STOP": {
      "value": null,
      "probability": 0.0
    },
    "START": {
      "value": null,
      "probability": 0.0
    },
    "ENGINE_STOP": {
      "value": null,
      "probability": 0.0
    },
    "ENGINE_START": {
      "value": null,
      "probability": 0.0
    },
    "CRANK_REQUEST": {
      "value": null,
      "probability": 0.0
    },
    "STOP_IN_PITLANE_TOGGLE_ON": {
      "value": null,
      "probability": 0.0
    },
    "STOP_IN_PITLANE_TOGGLE_OFF": {
      "value": null,
      "probability": 0.0
    },
    "SPEED_BY_RC_TOGGLE_ON": {
      "value": null,
      "probability": 0.0
    },
    "SPEED_BY_RC_TOGGLE_OFF": {
      "value": null,
      "probability": 0.0
    },
    "FLAG_BY_RC_TOGGLE_ON": {
      "value": null,
      "probability": 0.0
    },
    "FLAG_BY_RC_TOGGLE_OFF": {
      "value": null,
      "probability": 0.0
    },
    "INTO_PITLANE_TOGGLE_ON": {
      "value": null,
      "probability": 0.0
    },
    "INTO_PITLANE_TOGGLE_OFF": {
      "

In [6]:
is_command_classification_correct(llm_answer = answer, correct_answer = y[75], probability_threshold = 0.9)

('CLASSIFICATION_CORRECT', 'SPEED_REQUEST', 50, 1.0)

### Process all the data

In [7]:
# ChatGPT-4o, GPT-4o, o3-mini, GPT-4 Turbo, GPT-3.5 Turbo
models = ["chatgpt-4o-latest", "gpt-4o-2024-08-06", "o3-mini-2025-01-31", "gpt-4-turbo-2024-04-09", "gpt-3.5-turbo-0125"]
results = {}

for model in models:
  llm_answers = []
  predictions = {
      'Predicted request': [],
      'Predicted value': [],
      'Classification Result': [],
      'Probability': [],
      'Prompt processing time': []
  }

  probability_threshold = 0.80

  for i in range(len(X)):
      start_time = time.time() 
      completion = client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},
      messages=[
      {"role": "system",  "content": get_prompt_template()},
      {"role": "user",  "content": X[i][0]}
        ]
      )
      end_time = time.time()
      prompt_processing_time = end_time - start_time

      answer = completion.choices[0].message.content
      llm_answers.append(answer)
      classification_result = is_command_classification_correct(answer, y[i], probability_threshold)
      predictions['Classification Result'].append(classification_result[0])
      predictions['Prompt processing time'].append(prompt_processing_time)

      if classification_result[0] != "WRONG_OUTPUT_FORMAT":
          predictions['Predicted request'].append(classification_result[1])
          predictions['Predicted value'].append(classification_result[2])
          predictions['Probability'].append(classification_result[3])
      else:
          predictions['Predicted request'].append(None)
          predictions['Predicted value'].append(None)
          predictions['Probability'].append(None)


  predictions = pd.DataFrame(predictions)
  accuracy = predictions[predictions['Classification Result'] == "CLASSIFICATION_CORRECT"].shape[0] / predictions.shape[0] 
  results[model] = {"Accuracy" : accuracy, "Predictions" : predictions, "LLM Answers" : llm_answers}

with open("results_openai.pkl", "wb") as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
print("Models' accuracy")
for model in models:
    print("{}: {:.2f}%".format(model, results[model]["Accuracy"] * 100))

Models' accuracy
chatgpt-4o-latest: 76.47%
gpt-4o-2024-08-06: 74.12%
o3-mini-2025-01-31: 85.88%
gpt-4-turbo-2024-04-09: 81.18%
gpt-3.5-turbo-0125: 57.65%


In [9]:
print("Models' average prompt processing time")
for model in models:
    print("{}: {:.2f}s".format(model, results[model]["Predictions"].loc[:, 'Prompt processing time'].mean()))

Models' average prompt processing time
chatgpt-4o-latest: 3.38s
gpt-4o-2024-08-06: 4.16s
o3-mini-2025-01-31: 21.08s
gpt-4-turbo-2024-04-09: 15.73s
gpt-3.5-turbo-0125: 5.12s
